# Notebook 2: Feature Engineering

**Goal:** Build the feature set that drives the YakOS projection model.

**Input:** `yakos_historical_master.parquet`

**Output:** `yakos_features.parquet` — one row per player per slate, all features + actual DK FP as target

## Features computed per player per slate
- Rolling averages (last 5, 10, 20 game DK FP — weighted recency)
- Rolling minutes (last 5, 10 game average)
- Minutes stability (std dev over last 10 games)
- Pace environment (Vegas game total)
- Spread impact (blowout risk)
- Defense vs Position (DvP) — opponent's DK FP allowed per position over last 15 games
- Home/Away split
- Rest days (back-to-back flag, days since last game)
- Usage proxy (player's share of team FP)
- Salary
- Consensus projections (Tank01, RG)
- Recent trend (last 3 game FP vs season average)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_PATH  = 'yakos_historical_master.parquet'
OUTPUT_PATH = 'yakos_features.parquet'

master = pd.read_parquet(INPUT_PATH)
master['game_date'] = pd.to_datetime(master['game_date'])
master = master.sort_values(['player_name', 'game_date']).reset_index(drop=True)

print(f'Master shape: {master.shape}')
print(f'Date range: {master["game_date"].min()} to {master["game_date"].max()}')
master.head()

## Rolling FP Averages

In [ ]:
def rolling_fp(df, windows=(5, 10, 20)):
    """Compute rolling mean DK FP for each player. Uses shift(1) to avoid leakage."""
    df = df.copy()
    grp = df.groupby('player_name')['dk_fp_actual']
    for w in windows:
        df[f'rolling_fp_{w}'] = grp.transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).mean()
        )
    return df


master = rolling_fp(master)
print('Rolling FP columns added:', [c for c in master.columns if c.startswith('rolling_fp')])

## Rolling Minutes & Minutes Stability

In [ ]:
def rolling_minutes(df, windows=(5, 10, 20)):
    """Compute rolling mean and std of minutes for each player."""
    df = df.copy()
    if 'minutes' not in df.columns:
        for w in windows:
            df[f'rolling_min_{w}'] = np.nan
        df['min_std_10'] = np.nan
        return df

    grp = df.groupby('player_name')['minutes']
    for w in windows:
        df[f'rolling_min_{w}'] = grp.transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).mean()
        )
    # Minutes stability: std dev over last 10 games (high variance = risky)
    df['min_std_10'] = grp.transform(
        lambda s: s.shift(1).rolling(10, min_periods=2).std()
    )
    return df


master = rolling_minutes(master)
print('Minutes columns added.')

## Rest Days & Back-to-Back Flag

In [ ]:
def add_rest_features(df):
    """Compute days since last game and back-to-back flag."""
    df = df.copy()
    df = df.sort_values(['player_name', 'game_date'])
    grp = df.groupby('player_name')['game_date']
    df['days_rest'] = grp.diff().dt.days.fillna(7).clip(upper=14)
    df['b2b'] = (df['days_rest'] == 1).astype(int)
    return df


master = add_rest_features(master)
print(f'Back-to-back games: {master["b2b"].sum()}')

## Defense vs Position (DvP)

In [ ]:
def add_dvp(df, window=15):
    """
    Compute opponent's DK FP allowed per position over the last `window` games.
    A high DvP means the opponent lets that position score more — good matchup.
    """
    df = df.copy()
    if 'opponent' not in df.columns:
        df['dvp'] = np.nan
        return df

    df['primary_pos'] = df['pos'].str.split('/').str[0].str.strip()

    # For each date, compute the rolling FP allowed by each team per position
    # using only games BEFORE this date (shift to avoid leakage)
    df = df.sort_values('game_date')

    dvp_values = []
    for i, row in df.iterrows():
        opp   = row.get('opponent', '')
        pos   = row.get('primary_pos', '')
        date  = row['game_date']
        if not opp or not pos:
            dvp_values.append(np.nan)
            continue

        # Games where this opponent was the defending team and the player's position scored
        mask = (
            (df['team'] == opp) &
            (df['primary_pos'] == pos) &
            (df['game_date'] < date)
        )
        recent = df.loc[mask].nlargest(window, 'game_date')
        dvp_values.append(recent['dk_fp_actual'].mean() if len(recent) > 0 else np.nan)

    df['dvp'] = dvp_values
    return df


# NOTE: add_dvp can be slow on large datasets. Consider vectorising for production.
master = add_dvp(master)
print(f'DvP computed. Non-null: {master["dvp"].notna().sum()}')

## Home/Away Split

In [ ]:
def add_home_away(df):
    """Infer home/away from opponent field (if available)."""
    df = df.copy()
    # Convention: if opponent has '@' prefix, player is away
    if 'opponent' in df.columns:
        df['home'] = (~df['opponent'].str.startswith('@')).astype(int)
    else:
        df['home'] = np.nan
    return df


master = add_home_away(master)

## Usage Proxy & Recent Trend

In [ ]:
def add_usage_and_trend(df):
    """Compute usage proxy and recent trend vs season average."""
    df = df.copy()

    # Usage proxy: player's share of team's total FP per game (prior games only)
    df['team_fp'] = df.groupby(['team', 'game_date'])['dk_fp_actual'].transform('sum')
    df['usage_proxy'] = df['dk_fp_actual'] / df['team_fp'].replace(0, np.nan)

    # Rolling usage (last 10 games, shifted)
    df['rolling_usage_10'] = df.groupby('player_name')['usage_proxy'].transform(
        lambda s: s.shift(1).rolling(10, min_periods=1).mean()
    )

    # Season average FP (all prior games)
    df['season_avg_fp'] = df.groupby('player_name')['dk_fp_actual'].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    # Recent trend: last 3 game avg vs season avg (positive = hot, negative = cold)
    df['recent_fp_3'] = df.groupby('player_name')['dk_fp_actual'].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
    df['trend'] = df['recent_fp_3'] - df['season_avg_fp']

    return df


master = add_usage_and_trend(master)
print('Usage proxy and trend features added.')

## Value Score

In [ ]:
def add_value_score(df):
    """Compute value score: proj FP per $1K salary."""
    df = df.copy()
    if 'tank01_proj' in df.columns and 'salary' in df.columns:
        df['value_score'] = df['tank01_proj'] / (df['salary'] / 1000).replace(0, np.nan)
    else:
        df['value_score'] = np.nan
    return df


master = add_value_score(master)

## Save Features Parquet

In [ ]:
# Drop rows with no actual FP (current/future slates — no target available)
features_df = master.dropna(subset=['dk_fp_actual']).copy()

# Feature columns used by the models
FEATURE_COLS = [
    'salary', 'tank01_proj', 'rg_proj',
    'rolling_fp_5', 'rolling_fp_10', 'rolling_fp_20',
    'rolling_min_5', 'rolling_min_10', 'min_std_10',
    'dvp', 'vegas_total', 'vegas_spread',
    'home', 'b2b', 'days_rest',
    'rolling_usage_10', 'trend', 'value_score',
]

# Check availability
available = [c for c in FEATURE_COLS if c in features_df.columns]
missing   = [c for c in FEATURE_COLS if c not in features_df.columns]
print(f'Available features ({len(available)}): {available}')
print(f'Missing features ({len(missing)}): {missing}')

features_df.to_parquet(OUTPUT_PATH, index=False)
print(f'\nFeatures saved: {len(features_df)} rows, {len(features_df.columns)} columns → {OUTPUT_PATH}')